In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn import metrics as metric
import matplotlib.pyplot as plt 
from scipy.interpolate import interp1d
from scipy.interpolate import make_interp_spline, BSpline
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,roc_auc_score,recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer
import scipy
from sklearn.feature_selection import chi2
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectKBest



## Load the Data

In [ ]:

df = pd.read_csv("C:/Users/RoyaYousefi/Desktop/example_export_3.csv.csv")
df.head()
print(df['CT_occlusion_loc'])
df['CT_occlusion_loc'].shape




## EDA

In [ ]:
df.info()
df.describe()
df.columns


In [ ]:
df[["CT_occlusion_loc1","CT_occlusion_loc2","CT_occlusion_loc3"]] = df["CT_occlusion_loc"].str.split(";",expand=True)

In [ ]:
df["CT_occlusion_loc1"].shape
print(df["CT_occlusion_loc1"].value_counts())


In [ ]:
print(df["CT_occlusion_loc2"].value_counts())


In [ ]:
df["CT_occlusion_loc"].isna().sum()

In [ ]:
LVO_loc = ["CT_occlusion_loc1","CT_occlusion_loc2","CT_occlusion_loc3"]
LVO_list = ['1','2','4','6']
def LVO(row):
    for i in LVO_loc:
        if row[i] in LVO_list:
            return 1
        
    return 0
        
df["LVO/Non_LVO"] = df.apply(LVO, axis=1)

In [ ]:

print(df["LVO/Non_LVO"].value_counts())

In [ ]:
df.columns

for col in df.columns:
    lvo_sel = df["LVO/Non_LVO"] == 1
    non_lvo_sel = df["LVO/Non_LVO"] == 0
    data_available_lvo    = round((1.0 - float(df[col][lvo_sel].isna().sum()) / float(df[col][lvo_sel].size)) * 100.0)
    data_available_nonlvo = round((1.0 - float(df[col][non_lvo_sel].isna().sum()) / float(df[col][non_lvo_sel].size)) * 100.0)

    print(col, ": LVO data available: ", df[col][lvo_sel].size - df[col][lvo_sel].isna().sum()," (", data_available_lvo ,"%); data available_nonlvo ",df[col][non_lvo_sel].size - df[col][non_lvo_sel].isna().sum()," (", data_available_nonlvo, '%)')

In [ ]:
df["history"].isna().sum()

In [ ]:

medical_History = {"1":"Stroke","2":"Hemorrhagic Stroke","3":"Brain tumor","4":"Operation on the head",
                   "5":"Hypertension","6":"Diabetes","7":"Hypercholesterolemie",
                   "8":"Atrial fibrillation","9":"Migraine","10":"Epilepsie",
                   "11":"Serious skull brain injury","12":"Others","13":"TIA"}


df["history"] = df["history"].fillna("0").astype(str)

for index,mh in medical_History.items():
    df["history-" + mh] = df["history"].apply(lambda row: int(index in row.split(";")))
    

In [ ]:
print(df.columns.tolist())

In [ ]:


column_selection_for_cramers_v = ['history-Operation on the head', "history-Stroke", "history-Brain tumor", "history-Hemorrhagic Stroke",
       'history-Hypertension', 'history-Diabetes', 
       'history-Hypercholesterolemie', 'history-Atrial fibrillation', 
       'history-Migraine', 'history-Epilepsie', 'history-Serious skull brain injury', 
       'history-Others', 'history-TIA','LVO/Non_LVO']

def cramers_v(column1, column2):
    # expected input: 2 pandas dataframe columns
    # output: 1 value of cramers v
    # reference: https://
    contingency_table = pd.crosstab(column1, column2)
    chi2_stat, p, dof, ex = stats.chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v = np.sqrt(chi2_stat / (n * min_dim))
    return cramers_v

cramers_df = pd.DataFrame([])
for row in column_selection_for_cramers_v:
    list_temp = []
    for col in column_selection_for_cramers_v:
        list_temp.append(cramers_v(df[row], df[col]))
    cramers_df[row] = list_temp
cramers_df.index = column_selection_for_cramers_v


In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(cramers_df, annot=True, linewidths=1, cmap="YlGnBu", cbar_kws={'label': 'Cramér\'s V'})
plt.title("Heatmap of Cramér's V values")
plt.show()

In [ ]:


column_selection_for_ROC = ['history-Operation on the head', "history-Stroke", "history-Brain tumor", "history-Hemorrhagic Stroke",
       'history-Hypertension', 'history-Diabetes', 
       'history-Hypercholesterolemie', 'history-Atrial fibrillation', 
       'history-Migraine', 'history-Epilepsie', 'history-Serious skull brain injury', 
       'history-Others', 'history-TIA']



def ROC_Medical(df):
    n_row = 4
    n_cols = (len(column_selection_for_ROC) + n_row - 1) // n_row
    fig, axs = plt.subplots(n_row, n_cols, figsize=(5 * n_cols, 5 * n_row))

    axs = axs.flatten()

    for count, feature in enumerate(column_selection_for_ROC):
        df_feature = df.dropna(subset=[feature]).copy()
        
        
        if df_feature.empty:
            continue

        df_feature[feature] = df_feature[feature].astype(float)

        x = df_feature[feature]
        y = df_feature["LVO/Non_LVO"]

        
        #if np.median(x[y==1]) > np.median( x[y==0]):
            #fpr, tpr, thresholds = roc_curve(y, x)
        #else:
        fpr, tpr, thresholds = roc_curve(y, x)
        roc_auc = round(auc(fpr, tpr), 2) 

      

        lvo_sel = df["LVO/Non_LVO"] == 1
        non_lvo_sel = df["LVO/Non_LVO"] == 0
        lvo_percentage = round((1.0 - float(df[feature][lvo_sel].isna().sum()) / float(df[feature][lvo_sel].size)) * 100.0)
        nonlvo_percentage = round((1.0 - float(df[feature][non_lvo_sel].isna().sum()) / float(df[feature][non_lvo_sel].size)) * 100.0)
        

        
        ax = axs[count]
        ax.plot(fpr, tpr, label=f"ROC Curve for {feature} (AUC={roc_auc})")
        ax.plot([0., 1.], [0., 1.], 'k--')
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve for {feature}\nNon-LVO: {nonlvo_percentage}%, LVO: {lvo_percentage}%")
        ax.legend()

    for idx in range(len(column_selection_for_ROC), len(axs)):
        fig.delaxes(axs[idx])

    plt.tight_layout()
    plt.show()



ROC_Medical(df)

In [ ]:
NIHSS_columns_selection_for_cramers = ["NIHSS_1A","NIHSS_1B","NIHSS_1C","NIHSS_2",
                 "NIHSS_3","NIHSS_4","NIHSS_5A","NIHSS_5B",
                 "NIHSS_6A","NIHSS_6B","NIHSS_7","NIHSS_8",
                 "NIHSS_9","NIHSS_10","NIHSS_11","NIHSS","LVO/Non_LVO"]

def cramers_v_NIHSS(column1, column2):
    # expected input: 2 pandas dataframe columns
    # output: 1 value of cramers v
    # reference: https://
    contingency_table = pd.crosstab(column1, column2)
    chi2_stat, p, dof, ex = stats.chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v_NIHSS = np.sqrt(chi2_stat / (n * min_dim))
    return cramers_v_NIHSS


cramers_df_NIHSS = pd.DataFrame([])
for row in NIHSS_columns_selection_for_cramers:
    list_temp_NIHSS = []
    for col in NIHSS_columns_selection_for_cramers:
        list_temp_NIHSS.append(cramers_v_NIHSS(df[row], df[col]))
    cramers_df_NIHSS[row] = list_temp_NIHSS
cramers_df_NIHSS.index = NIHSS_columns_selection_for_cramers
print(cramers_df_NIHSS)

In [ ]:
df[NIHSS_columns_selection_for_cramers].isna().sum()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(cramers_df_NIHSS, annot=True, linewidths=1, cmap="YlGnBu", cbar_kws={'label': 'Cramér\'s V'})
plt.title("Heatmap of Cramér's V values")
plt.show()

In [ ]:


NIHSS_columns_selection_for_ROC = ["NIHSS_1A","NIHSS_1B","NIHSS_1C","NIHSS_2",
                 "NIHSS_3","NIHSS_4","NIHSS_5A","NIHSS_5B",
                 "NIHSS_6A","NIHSS_6B","NIHSS_7","NIHSS_8",
                 "NIHSS_9","NIHSS_10","NIHSS_11","NIHSS"]



def ROC_NIHSS(df):
    n_row = 4
    n_cols = (len(NIHSS_columns_selection_for_ROC) + n_row - 1) // n_row
    fig, axs = plt.subplots(n_row, n_cols, figsize=(5 * n_cols, 5 * n_row))

    axs = axs.flatten()

    for count, feature in enumerate(NIHSS_columns_selection_for_ROC):
        df_feature = df.dropna(subset=[feature]).copy()
        
        
        if df_feature.empty:
            continue

        df_feature[feature] = df_feature[feature].astype(float)

        x = df_feature[feature]
        y = df_feature["LVO/Non_LVO"]

        
        #if np.median(x[y==1]) > np.median( x[y==0]):
            #fpr, tpr, thresholds = roc_curve(y, x)
        #else:
        fpr, tpr, thresholds = roc_curve(y, x)
        roc_auc = round(auc(fpr, tpr), 2) 

      

        lvo_sel = df["LVO/Non_LVO"] == 1
        non_lvo_sel = df["LVO/Non_LVO"] == 0
        lvo_percentage = round((1.0 - float(df[feature][lvo_sel].isna().sum()) / float(df[feature][lvo_sel].size)) * 100.0)
        nonlvo_percentage = round((1.0 - float(df[feature][non_lvo_sel].isna().sum()) / float(df[feature][non_lvo_sel].size)) * 100.0)
        

        
        ax = axs[count]
        ax.plot(fpr, tpr, label=f"ROC Curve for {feature} (AUC={roc_auc})")
        ax.plot([0., 1.], [0., 1.], 'k--')
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve for {feature}\nNon-LVO: {nonlvo_percentage}%, LVO: {lvo_percentage}%")
        ax.legend()

    for idx in range(len(NIHSS_columns_selection_for_ROC), len(axs)):
        fig.delaxes(axs[idx])

    plt.tight_layout()
    plt.show()



ROC_NIHSS(df)

In [ ]:
df["LVO/Non_LVO"].isna().sum()

In [ ]:
df['contra_delta'].isna().sum()

In [ ]:

if set(df["LVO/Non_LVO"]) != {0, 1}:
    print("Target variable contains non-binary value")

In [ ]:
df["LVO/Non_LVO"].value_counts()

In [ ]:
print(df["LVO/Non_LVO"].unique())

In [ ]:


EEG_features = ['contra_delta', 'contra_theta', 'contra_alpha', 'contra_beta', 
                'contra_DAR', 'contra_TAR', 'contra_DTABR', 'contra_pdbsi_delta',
                'contra_pdbsi_theta', 'contra_pdbsi_alpha', 'contra_pdbsi_beta', 
                'contra_pdbsi_total_1_18','ipsi_delta', 'ipsi_theta', 'ipsi_alpha', 
                'ipsi_beta', 'ipsi_DAR', 'ipsi_TAR', 'ipsi_DTABR', 'ipsi_pdbsi_delta', 
                'ipsi_pdbsi_theta', 'ipsi_pdbsi_alpha', 'ipsi_pdbsi_beta', 'ipsi_pdbsi_total_1_18']

def fill_Nan(df):
    n_row = 4
    n_cols = (len(EEG_features) + n_row - 1) // n_row
    fig, axs = plt.subplots(n_row, n_cols, figsize=(5 * n_cols, 5 * n_row))

    axs = axs.flatten()

    for count, feature in enumerate(EEG_features):
        df_feature = df.dropna(subset=[feature]).copy()
        
        
        if df_feature.empty:
            continue

        df_feature[feature] = df_feature[feature].astype(float)

        x = df_feature[feature]
        y = df_feature["LVO/Non_LVO"]

        
        if np.median(x[y==1]) > np.median( x[y==0]):
            fpr, tpr, thresholds = roc_curve(y, x)
        else:
            fpr, tpr, thresholds = roc_curve(y, -x)
        roc_auc = round(auc(fpr, tpr), 2) 

      

        lvo_sel = df["LVO/Non_LVO"] == 1
        non_lvo_sel = df["LVO/Non_LVO"] == 0
        lvo_percentage = round((1.0 - float(df[feature][lvo_sel].isna().sum()) / float(df[feature][lvo_sel].size)) * 100.0)
        nonlvo_percentage = round((1.0 - float(df[feature][non_lvo_sel].isna().sum()) / float(df[feature][non_lvo_sel].size)) * 100.0)
        

        
        ax = axs[count]
        ax.plot(fpr, tpr, label=f"ROC Curve for {feature} (AUC={roc_auc})")
        ax.plot([0., 1.], [0., 1.], 'k--')
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve for {feature}\nNon-LVO: {nonlvo_percentage}%, LVO: {lvo_percentage}%")
        ax.legend()

    for idx in range(len(EEG_features), len(axs)):
        fig.delaxes(axs[idx])

    plt.tight_layout()
    plt.show()



fill_Nan(df)


In [ ]:
df[NIHSS_columns_selection_for_ROC].isna().sum()

In [ ]:
df[column_selection_for_ROC].isna().sum()

In [ ]:
EEG_features = ['contra_delta', 'contra_theta', 'contra_alpha', 'contra_beta', 
                'contra_DAR', 'contra_TAR', 'contra_DTABR', 'contra_pdbsi_delta',
                'contra_pdbsi_theta', 'contra_pdbsi_alpha', 'contra_pdbsi_beta', 
                'contra_pdbsi_total_1_18','ipsi_delta', 'ipsi_theta', 'ipsi_alpha', 
                'ipsi_beta', 'ipsi_DAR', 'ipsi_TAR', 'ipsi_DTABR', 'ipsi_pdbsi_delta', 
                'ipsi_pdbsi_theta', 'ipsi_pdbsi_alpha', 'ipsi_pdbsi_beta', 'ipsi_pdbsi_total_1_18']

def check_nan(df, EEG_features):
    for feature in EEG_features:
        nan_count = df[feature].isna().sum()
        if nan_count > 0:
            print(f"{feature} has {nan_count} missing values")
            
            
            if not df[feature].mode().empty:
                mode_value = df[feature].mode().iloc[0]
                
                df[feature] = df[feature].fillna(mode_value)
            else:
                print(f"{feature} has no mode value to fill NaNs.")
            
            nan_count_after_fill = df[feature].isna().sum()
            print(f"After filling, {feature} has {nan_count_after_fill} missing values")


check_nan(df, EEG_features)


In [ ]:

EEG_features = ['contra_delta', 'contra_theta', 'contra_alpha', 'contra_beta', 
                'contra_DAR', 'contra_TAR', 'contra_DTABR', 'contra_pdbsi_delta',
                'contra_pdbsi_theta', 'contra_pdbsi_alpha', 'contra_pdbsi_beta', 
                'contra_pdbsi_total_1_18','ipsi_delta', 'ipsi_theta', 'ipsi_alpha', 
                'ipsi_beta', 'ipsi_DAR', 'ipsi_TAR', 'ipsi_DTABR', 'ipsi_pdbsi_delta', 
                'ipsi_pdbsi_theta', 'ipsi_pdbsi_alpha', 'ipsi_pdbsi_beta', 'ipsi_pdbsi_total_1_18']


X = df.drop("LVO/Non_LVO", axis=1)
scaler = MinMaxScaler()
X_copy = X.copy(deep=False)    
X_copy_scaled = scaler.fit_transform(X_copy[EEG_features])
X_copy_scaled= pd.DataFrame(X_copy_scaled, columns=list(EEG_features))
y = df["LVO/Non_LVO"]


In [ ]:
X_copy_scaled.shape

In [ ]:
X_copy_scaled.head()

In [ ]:
df["LVO/Non_LVO"].shape

### EEG_features selection

In [ ]:

function_list = [mutual_info_classif, chi2] 


y= df["LVO/Non_LVO"]
scores = pd.DataFrame(columns=X_copy_scaled.columns) 


for f in range(len(function_list)):
     selector= SelectKBest(function_list[f], k=11).fit(X_copy_scaled,y)
     score = selector.scores_
     scores.loc[len(scores)] = score

 
features = scores.transpose()
features=features.rename(columns={0:"mutual_info_classif",1:"chi2"})
features.sort_values(by="chi2" ,ascending=False, inplace=True)
features.head() 


sns.set(style="whitegrid")  
sns.lineplot(data=features, x=features.index, y="mutual_info_classif",color='green')
sns.lineplot(data=features, x=features.index, y="chi2",color='red')
plt.xticks(rotation=90, fontsize=8)  
plt.show()
print(features.head(20))

### Medical features selection 

In [ ]:
Medical_features = ['history-Operation on the head', "history-Stroke", "history-Brain tumor", "history-Hemorrhagic Stroke",
       'history-Hypertension', 'history-Diabetes', 
       'history-Hypercholesterolemie', 'history-Atrial fibrillation', 
       'history-Migraine', 'history-Epilepsie', 'history-Serious skull brain injury', 
       'history-Others', 'history-TIA']

X = df.drop("LVO/Non_LVO", axis=1)
scaler = MinMaxScaler()
X_copy = X.copy(deep=False)    
X_copy_scaled_medical = scaler.fit_transform(X_copy[Medical_features])
X_copy_scaled_medical= pd.DataFrame(X_copy_scaled_medical, columns=list(Medical_features))
y = df["LVO/Non_LVO"]

In [ ]:

function_list = [mutual_info_classif, chi2] 


y= df["LVO/Non_LVO"]
scores_df_medical = pd.DataFrame(columns=X_copy_scaled_medical.columns) 


for f in range(len(function_list)):
     selector_medical= SelectKBest(function_list[f], k=11).fit(X_copy_scaled_medical,y)
     score_medical = selector_medical.scores_
     scores_df_medical.loc[len(scores_df_medical)] = score_medical

 
features_medical = scores_df_medical.transpose()
features_medical=features_medical.rename(columns={0:"mutual_info_classif",1:"chi2"})
features_medical.sort_values(by="chi2",ascending=False, inplace=True)
features_medical.head() 

 

sns.set(style="whitegrid")  
sns.lineplot(data=features_medical, x=features_medical.index, y="mutual_info_classif",color='green')
sns.lineplot(data=features_medical, x=features_medical.index, y="chi2",color='red')
plt.xticks(rotation=90, fontsize=8)  
plt.show()
print(features_medical.head(20))

In [ ]:
features_medical = scores_df_medical.transpose()
features_medical=features_medical.rename(columns={0:"mutual_info_classif",1:"chi2"})
features_medical.sort_values(by="chi2",ascending=False, inplace=True)
features_medical.head(6)

### NIHSS feature selection

In [ ]:

NIHSS_features_selection = ["NIHSS_1A","NIHSS_1B","NIHSS_1C","NIHSS_2",
                 "NIHSS_3","NIHSS_4","NIHSS_5A","NIHSS_5B",
                 "NIHSS_6A","NIHSS_6B","NIHSS_7","NIHSS_8",
                 "NIHSS_9","NIHSS_10","NIHSS_11","NIHSS"]

X = df.drop("LVO/Non_LVO", axis=1)
scaler = MinMaxScaler()
X_copy = X.copy(deep=False)    
X_copy_scaled_NIHSS = scaler.fit_transform(X_copy[NIHSS_features_selection])
X_copy_NIHSS= pd.DataFrame(X_copy_scaled_NIHSS, columns=list(NIHSS_features_selection))
y = df["LVO/Non_LVO"]

In [ ]:

function_list = [mutual_info_classif, chi2] 


y= df["LVO/Non_LVO"]
scores_df_NIHSS = pd.DataFrame(columns=X_copy_NIHSS.columns) 


for f in range(len(function_list)):
     selector_NIHSS = SelectKBest(function_list[f], k=11).fit(X_copy_NIHSS,y)
     score_NIHSS = selector_NIHSS.scores_
     scores_df_NIHSS.loc[len(scores_df_NIHSS)] = score_NIHSS

 
features_NIHSS = scores_df_NIHSS.transpose()
features_NIHSS=features_NIHSS.rename(columns={0:"mutual_info_classif",1:"chi2"})
features_NIHSS.sort_values(by="chi2" ,ascending=False, inplace=True)
features_NIHSS.head() 


sns.set(style="whitegrid")  
sns.lineplot(data=features_NIHSS, x=features_NIHSS.index, y="mutual_info_classif",color='green')
sns.lineplot(data=features_NIHSS, x=features_NIHSS.index, y="chi2",color='red')
plt.xticks(rotation=90, fontsize=8)  
plt.show()
print(features_NIHSS.head(20))

### Modeling

### Model on EEG Features

In [ ]:
features.head(11)

In [ ]:
EEG_Best = ["ipsi_alpha","ipsi_DTABR","ipsi_DAR","ipsi_delta","contra_alpha","contra_DTABR",
            "contra_DAR","contra_delta"]

In [ ]:

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_index, test_index = next(sss.split(df[EEG_Best], y))

X_train, X_test = df[EEG_Best].iloc[train_index], df[EEG_Best].iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]


smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
    

In [ ]:
classifiers = [SVC(),LogisticRegression(),DecisionTreeClassifier()]
 
classifiers_names = ['SVM','Logistic Regression','Decision Tree']
 
 
SVM_params = params = {'gamma': ['scale', 'auto'],'kernel': ['rbf', 'poly'],'C' : [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],"class_weight":["balanced"],
                       'coef0':  [0, 0.1, 0.5, 1, 2],'break_ties': [True], 'cache_size': [100, 200, 500, 1000]}
#['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma',
# 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose']
 
 
LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False]
}
 
 
#'C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'multi_class',
# 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start']
 
DT_params = params = {'max_depth':list(range(10, 15)),"class_weight":["balanced"],'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0]}
#'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf',
# 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].
 
 
classifiers_Params = {'SVM': SVM_params,'Logistic Regression':LR_params, 'Decision Tree':DT_params}
 
results = pd.DataFrame(columns=['estimators','accuracy','recall_score','confusion_matrix','roc_curve'])
 
for i,clf in enumerate(classifiers):
    model = clf
 
    clf_gs = GridSearchCV(model, classifiers_Params[classifiers_names[i]], cv = 5, scoring='accuracy')
   
    clf_gs.fit(X_train, y_train)
 
    print(clf_gs.best_estimator_.get_params())
 
    pred = clf_gs.predict(X_test)
    acs = accuracy_score(y_test, pred)
    rec = recall_score(y_test, pred)
    cm = confusion_matrix(y_test,pred)
    auc = roc_auc_score(y_test, clf_gs.predict_proba(X_test)[:, 1] if hasattr(clf_gs, "predict_proba") else pred)
       
    results.loc[i] = [classifiers_names[i],acs,rec,cm,auc]
 
results.head(5)

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Handle class imbalance
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Define classifiers and hyperparameters
classifiers = [
    SVC(probability=True),
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
    XGBClassifier(use_label_encoder=False, eval_metric='logloss')
]
classifiers_names = ['SVM', 'Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']

SVM_params = {
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],
    'max_iter': [100, 200, 500, 1000]
}

DT_params = {
    'max_depth': list(range(5, 20)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]
}

RF_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'class_weight': ['balanced'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

GB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'subsample': [0.8, 0.9, 1.0]
}

XGB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 10, 20]
}

classifiers_Params = {
    'SVM': SVM_params,
    'Logistic Regression': LR_params,
    'Decision Tree': DT_params,
    'Random Forest': RF_params,
    'Gradient Boosting': GB_params,
    'XGBoost': XGB_params
}


results = pd.DataFrame(columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])


for i, clf in enumerate(classifiers):
    model = clf
    param_grid = classifiers_Params[classifiers_names[i]]
    clf_gs = RandomizedSearchCV(model, param_distributions=param_grid, cv=5, scoring='accuracy', error_score='raise', n_iter=50, random_state=42)
    
    clf_gs.fit(X_train, y_train)
    best_estimator = clf_gs.best_estimator_
    
    print(best_estimator.get_params())

    pred = best_estimator.predict(X_test)
    acs = accuracy_score(y_test, pred)
    rec = recall_score(y_test, pred)
    prec = precision_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    cm = confusion_matrix(y_test, pred)
    auc = roc_auc_score(y_test, best_estimator.predict_proba(X_test)[:, 1] if hasattr(best_estimator, "predict_proba") else pred)
    
    results.loc[i] = [classifiers_names[i], acs, rec, prec, f1, cm, auc]

print(results.head())


ensemble_model = VotingClassifier(estimators=[(name, clf) for name, clf in zip(classifiers_names, classifiers)], voting='soft')
ensemble_model.fit(X_train, y_train)
pred = ensemble_model.predict(X_test)
acs = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred)
prec = precision_score(y_test, pred)
f1 = f1_score(y_test, pred)
cm = confusion_matrix(y_test, pred)
auc = roc_auc_score(y_test, ensemble_model.predict_proba(X_test)[:, 1] if hasattr(ensemble_model, "predict_proba") else pred)

ensemble_results = pd.DataFrame([['Ensemble', acs, rec, prec, f1, cm, auc]], columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])
results = pd.concat([results, ensemble_results])
 
results.head(7)

### Model on Medical Features

In [ ]:
features_medical.head(11)

In [ ]:

# Define features and target variable
Medical_Best = ["history-Stroke", "history-TIA", "history-Others","history-Hypertension",
                "history-Atrial fibrillation", "history-Diabetes","history-Epilepsie",
                "history-Brain tumor","history-Serious skull brain injury",
                "history-Migraine","history-Hypercholesterolemie"]
 

# Splitting the dataset
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_index, test_index = next(sss.split(df[Medical_Best], y))
X_train, X_test = df[Medical_Best].iloc[train_index], df[Medical_Best].iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Handle class imbalance
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Define classifiers and hyperparameters
classifiers = [
    SVC(probability=True),
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
    XGBClassifier(use_label_encoder=False, eval_metric='logloss')
]
classifiers_names = ['SVM', 'Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']

SVM_params = {
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],
    'max_iter': [100, 200, 500, 1000]
}

DT_params = {
    'max_depth': list(range(5, 20)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]
}

RF_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'class_weight': ['balanced'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

GB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'subsample': [0.8, 0.9, 1.0]
}

XGB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 10, 20]
}

classifiers_Params = {
    'SVM': SVM_params,
    'Logistic Regression': LR_params,
    'Decision Tree': DT_params,
    'Random Forest': RF_params,
    'Gradient Boosting': GB_params,
    'XGBoost': XGB_params
}


results = pd.DataFrame(columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])


for i, clf in enumerate(classifiers):
    model = clf
    param_grid = classifiers_Params[classifiers_names[i]]
    clf_gs = RandomizedSearchCV(model, param_distributions=param_grid, cv=5, scoring='accuracy', error_score='raise', n_iter=50, random_state=42)
    
    clf_gs.fit(X_train, y_train)
    best_estimator = clf_gs.best_estimator_
    
    print(best_estimator.get_params())

    pred = best_estimator.predict(X_test)
    acs = accuracy_score(y_test, pred)
    rec = recall_score(y_test, pred)
    prec = precision_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    cm = confusion_matrix(y_test, pred)
    auc = roc_auc_score(y_test, best_estimator.predict_proba(X_test)[:, 1] if hasattr(best_estimator, "predict_proba") else pred)
    
    results.loc[i] = [classifiers_names[i], acs, rec, prec, f1, cm, auc]

print(results.head())


ensemble_model = VotingClassifier(estimators=[(name, clf) for name, clf in zip(classifiers_names, classifiers)], voting='soft')
ensemble_model.fit(X_train, y_train)
pred = ensemble_model.predict(X_test)
acs = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred)
prec = precision_score(y_test, pred)
f1 = f1_score(y_test, pred)
cm = confusion_matrix(y_test, pred)
auc = roc_auc_score(y_test, ensemble_model.predict_proba(X_test)[:, 1] if hasattr(ensemble_model, "predict_proba") else pred)

ensemble_results = pd.DataFrame([['Ensemble', acs, rec, prec, f1, cm, auc]], columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])
results = pd.concat([results, ensemble_results])
 
results.head(7)


In [ ]:
Medical_Best = ["history-Stroke","history-TIA","history-Others",
                "history-Epilepsie",
                "history-Brain tumor","history-Atrial fibrillation",
                "history-Migraine","history-Diabetes",
                "history-Serious skull brain injury","history-Hypertension","history-Hypercholesterolemie"]

In [ ]:

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_index, test_index = next(sss.split(df[Medical_Best], y))

X_train, X_test = df[Medical_Best].iloc[train_index], df[Medical_Best].iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]


smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
    

In [ ]:
classifiers = [SVC(),LogisticRegression(100),DecisionTreeClassifier()]

classifiers_names = ['SVM','Logistic Regression','Decision Tree']

SVM_params = {
  'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'break_ties': [True],
    'cache_size': [100, 200, 500, 1000]
}
#['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 
# 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'] 

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],'max_iter': [100, 200, 500, 1000]
}


#'C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'multi_class', 
# 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start'] 


DT_params = {
    'max_depth': list(range(10, 15)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]}

#'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 
# 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].


classifiers_Params = {'SVM': SVM_params,'Logistic Regression':LR_params, 'Decision Tree':DT_params}

results = pd.DataFrame(columns=['estimators','accuracy','recall_score','confusion_matrix','roc_curve'])
 
for i,clf in enumerate(classifiers):
    model = clf
 
    clf_gs = GridSearchCV(model, classifiers_Params[classifiers_names[i]], cv = 7, scoring='accuracy',error_score='raise')
   
    clf_gs.fit(X_train, y_train)

    print(clf_gs.best_estimator_.get_params())

    pred = clf_gs.predict(X_test)
    acs = accuracy_score(y_test, pred)
    rec = recall_score(y_test, pred)
    cm = confusion_matrix(y_test,pred)
    auc = roc_auc_score(y_test, clf_gs.predict_proba(X_test)[:, 1] if hasattr(clf_gs, "predict_proba") else pred)
        
    results.loc[i] = [classifiers_names[i],acs,rec,cm,auc]
 
results.head(5)

### Model on NIHSS features

In [ ]:
features_NIHSS.head(11)

In [ ]:
NIHSS_Best =["NIHSS_2","NIHSS_11","NIHSS_1B","NIHSS_9","NIHSS_4",
             "NIHSS_3","NIHSS_10","NIHSS","NIHSS_5B","NIHSS_8","NIHSS_1C"]


In [ ]:

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_index, test_index = next(sss.split(df[NIHSS_Best], y))

X_train, X_test = df[NIHSS_Best].iloc[train_index], df[NIHSS_Best].iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]


smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
    

In [ ]:
classifiers = [SVC(),LogisticRegression(),DecisionTreeClassifier()]

classifiers_names = ['SVM','Logistic Regression','Decision Tree']

SVM_params = params = { 'gamma': ['scale', 'auto'],'kernel': ['rbf'],'C' : [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],"class_weight":["balanced"],
                       'coef0':  [0, 0.1, 0.5, 1, 2],'break_ties': [True], 'cache_size': [100, 200, 500, 1000]}
#['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 
# 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'] 


LR_params =params = { 'dual': [False],
          'solver' : ['newton-cg', 'lbfgs', 'liblinear'],'penalty' : ['l2'],'C' : [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]}
#'C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'multi_class', 
# 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start'] 



DT_params = params = {'max_depth':list(range(10, 15))}
#'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 
# 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].


classifiers_Params = {'SVM': SVM_params,'Logistic Regression':LR_params, 'Decision Tree':DT_params}

results = pd.DataFrame(columns=['estimators','accuracy','recall_score','confusion_matrix','roc_curve'])
 
for i,clf in enumerate(classifiers):
    model = clf
 
    clf_gs = GridSearchCV(model, classifiers_Params[classifiers_names[i]], cv = 7, scoring='accuracy')
   
    clf_gs.fit(X_train, y_train)

    print(clf_gs.best_estimator_.get_params())

    pred = clf_gs.predict(X_test)
    acs = accuracy_score(y_test, pred)
    rec = recall_score(y_test, pred)
    cm = confusion_matrix(y_test,pred)
    auc = roc_auc_score(y_test, clf_gs.predict_proba(X_test)[:, 1] if hasattr(clf_gs, "predict_proba") else pred)
        
    results.loc[i] = [classifiers_names[i],acs,rec,cm,auc]
 
results.head(5)

In [ ]:
# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Handle class imbalance
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Define classifiers and hyperparameters
classifiers = [
    SVC(probability=True),
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
    XGBClassifier(use_label_encoder=False, eval_metric='logloss')
]
classifiers_names = ['SVM', 'Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']

SVM_params = {
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],
    'max_iter': [100, 200, 500, 1000]
}

DT_params = {
    'max_depth': list(range(5, 20)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]
}

RF_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'class_weight': ['balanced'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

GB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'subsample': [0.8, 0.9, 1.0]
}

XGB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 10, 20]
}

classifiers_Params = {
    'SVM': SVM_params,
    'Logistic Regression': LR_params,
    'Decision Tree': DT_params,
    'Random Forest': RF_params,
    'Gradient Boosting': GB_params,
    'XGBoost': XGB_params
}


results = pd.DataFrame(columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])


for i, clf in enumerate(classifiers):
    model = clf
    param_grid = classifiers_Params[classifiers_names[i]]
    clf_gs = RandomizedSearchCV(model, param_distributions=param_grid, cv=5, scoring='accuracy', error_score='raise', n_iter=50, random_state=42)
    
    clf_gs.fit(X_train, y_train)
    best_estimator = clf_gs.best_estimator_
    
    print(best_estimator.get_params())

    pred = best_estimator.predict(X_test)
    acs = accuracy_score(y_test, pred)
    rec = recall_score(y_test, pred)
    prec = precision_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    cm = confusion_matrix(y_test, pred)
    auc = roc_auc_score(y_test, best_estimator.predict_proba(X_test)[:, 1] if hasattr(best_estimator, "predict_proba") else pred)
    
    results.loc[i] = [classifiers_names[i], acs, rec, prec, f1, cm, auc]

print(results.head())


ensemble_model = VotingClassifier(estimators=[(name, clf) for name, clf in zip(classifiers_names, classifiers)], voting='soft')
ensemble_model.fit(X_train, y_train)
pred = ensemble_model.predict(X_test)
acs = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred)
prec = precision_score(y_test, pred)
f1 = f1_score(y_test, pred)
cm = confusion_matrix(y_test, pred)
auc = roc_auc_score(y_test, ensemble_model.predict_proba(X_test)[:, 1] if hasattr(ensemble_model, "predict_proba") else pred)

ensemble_results = pd.DataFrame([['Ensemble', acs, rec, prec, f1, cm, auc]], columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])
results = pd.concat([results, ensemble_results])
 
results.head(7)

### Combined features :EEG , NIHSS ,Medical history 

In [ ]:
df_EEG_Best = df[EEG_Best]
df_Medical_Best = df[Medical_Best]
df_NIHSS_Best = df[NIHSS_Best]


combined_df = pd.concat([df_EEG_Best, df_Medical_Best, df_NIHSS_Best], axis=1)
combined_df.head()


In [ ]:


sss_com = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_index_com, test_index_com= next(sss_com.split(combined_df, y))

X_train_com, X_test_com = combined_df.iloc[train_index_com], combined_df.iloc[test_index_com]
y_train_com, y_test_com = y.iloc[train_index_com], y.iloc[test_index_com]


smote = SMOTE(random_state=42)
X_train_com, y_train_com = smote.fit_resample(X_train_com, y_train_com)

In [ ]:
classifiers = [SVC(),LogisticRegression(100),DecisionTreeClassifier()]

classifiers_names = ['SVM','Logistic Regression','Decision Tree']

SVM_params = {
  'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}
#['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 
# 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'] 

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],'max_iter': [100, 200, 500, 1000]
}


#'C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'multi_class', 
# 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start'] 


DT_params = {
    'max_depth': list(range(10, 15)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]}

#'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 
# 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].


classifiers_Params = {'SVM': SVM_params,'Logistic Regression':LR_params, 'Decision Tree':DT_params}

results = pd.DataFrame(columns=['estimators','accuracy','recall_score','confusion_matrix','roc_curve'])
 
for i,clf in enumerate(classifiers):
    model = clf
 
    clf_gs = GridSearchCV(model, classifiers_Params[classifiers_names[i]], cv = 7, scoring='accuracy',error_score='raise')
   
    clf_gs.fit(X_train_com, y_train_com)

    print(clf_gs.best_estimator_.get_params())

    pred = clf_gs.predict(X_test_com)
    acs = accuracy_score(y_test_com, pred)
    rec = recall_score(y_test_com, pred)
    cm = confusion_matrix(y_test_com,pred)
    auc = roc_auc_score(y_test_com, clf_gs.predict_proba(X_test_com)[:, 1] if hasattr(clf_gs, "predict_proba") else pred)
        
    results.loc[i] = [classifiers_names[i],acs,rec,cm,auc]
 
results.head(5)

In [ ]:


classifiers = [
    SVC(probability=True),
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
    XGBClassifier(use_label_encoder=False, eval_metric='logloss')
]
classifiers_names = ['SVM', 'Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']

SVM_params = {
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],
    'max_iter': [100, 200, 500, 1000]
}

DT_params = {
    'max_depth': list(range(5, 20)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]
}

RF_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'class_weight': ['balanced'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

GB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'subsample': [0.8, 0.9, 1.0]
}

XGB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 10, 20]
}

classifiers_Params = {
    'SVM': SVM_params,
    'Logistic Regression': LR_params,
    'Decision Tree': DT_params,
    'Random Forest': RF_params,
    'Gradient Boosting': GB_params,
    'XGBoost': XGB_params
}


results = pd.DataFrame(columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])


for i, clf in enumerate(classifiers):
    model = clf
    param_grid = classifiers_Params[classifiers_names[i]]
    clf_gs = RandomizedSearchCV(model, param_distributions=param_grid, cv=7, scoring='accuracy', error_score='raise', n_iter=50, random_state=42)
    
    clf_gs.fit(X_train_com, y_train_com)
    best_estimator = clf_gs.best_estimator_
    
    print(best_estimator.get_params())

    pred = best_estimator.predict(X_test_com)
    acs = accuracy_score(y_test_com, pred)
    rec = recall_score(y_test_com, pred)
    prec = precision_score(y_test_com, pred)
    f1 = f1_score(y_test_com, pred)
    cm = confusion_matrix(y_test_com, pred)
    auc = roc_auc_score(y_test_com, best_estimator.predict_proba(X_test_com)[:, 1] if hasattr(best_estimator, "predict_proba") else pred)
    
    results.loc[i] = [classifiers_names[i], acs, rec, prec, f1, cm, auc]

print(results.head())


ensemble_model = VotingClassifier(estimators=[(name, clf) for name, clf in zip(classifiers_names, classifiers)], voting='soft')
ensemble_model.fit(X_train_com, y_train_com)
pred = ensemble_model.predict(X_test_com)
acs = accuracy_score(y_test_com, pred)
rec = recall_score(y_test_com, pred)
prec = precision_score(y_test_com, pred)
f1 = f1_score(y_test_com, pred)
cm = confusion_matrix(y_test_com, pred)
auc = roc_auc_score(y_test_com, ensemble_model.predict_proba(X_test_com)[:, 1] if hasattr(ensemble_model, "predict_proba") else pred)

ensemble_results = pd.DataFrame([['Ensemble', acs, rec, prec, f1, cm, auc]], columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])
results = pd.concat([results, ensemble_results])
 
results.head(7)

### Combine EEG with NIHSS

In [ ]:
df_EEG_Best = df[EEG_Best]
df_NIHSS_Best = df[NIHSS_Best]


combined_df_EN = pd.concat([df_EEG_Best, df_NIHSS_Best], axis=1)
combined_df_EN.head()

In [ ]:
import warnings
warnings.filterwarnings('ignore')

sss_com_EN = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_index_com_EN, test_index_com_EN= next(sss_com_EN.split( combined_df_EN, y))

X_train_com_EN, X_test_com_EN = combined_df_EN.iloc[train_index_com_EN], combined_df_EN.iloc[test_index_com_EN]
y_train_com_EN, y_test_com_EN = y.iloc[train_index_com_EN], y.iloc[test_index_com_EN]


smote = SMOTE(random_state=42)
X_train_com_EN, y_train_com_EN = smote.fit_resample(X_train_com_EN, y_train_com_EN)

In [ ]:
classifiers = [SVC(),LogisticRegression(100),DecisionTreeClassifier()]

classifiers_names = ['SVM','Logistic Regression','Decision Tree']

SVM_params = {
  'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}
#['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 
# 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'] 

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],'max_iter': [100, 200, 500, 1000]
}


#'C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'multi_class', 
# 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start'] 


DT_params = {
    'max_depth': list(range(10, 15)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]}

#'ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 
# 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].


classifiers_Params = {'SVM': SVM_params,'Logistic Regression':LR_params, 'Decision Tree':DT_params}

results = pd.DataFrame(columns=['estimators','accuracy','recall_score','confusion_matrix','roc_curve'])
 
for i,clf in enumerate(classifiers):
    model = clf
 
    clf_gs = GridSearchCV(model, classifiers_Params[classifiers_names[i]], cv = 7, scoring='accuracy',error_score='raise')
   
    clf_gs.fit(X_train_com_EN, y_train_com_EN)

    print(clf_gs.best_estimator_.get_params())

    pred = clf_gs.predict(X_test_com_EN)
    acs = accuracy_score(y_test_com_EN, pred)
    rec = recall_score(y_test_com_EN, pred)
    cm = confusion_matrix(y_test_com_EN,pred)
    auc = roc_auc_score(y_test_com_EN, clf_gs.predict_proba(X_test_com_EN)[:, 1] if hasattr(clf_gs, "predict_proba") else pred)
        
    results.loc[i] = [classifiers_names[i],acs,rec,cm,auc]
 
results.head(5)

In [ ]:

classifiers = [
    SVC(probability=True),
    LogisticRegression(max_iter=1000),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    GradientBoostingClassifier(),
    XGBClassifier(use_label_encoder=False, eval_metric='logloss')
]
classifiers_names = ['SVM', 'Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']

SVM_params = {
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf', 'poly'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'class_weight': ['balanced'],
    'coef0': [0, 0.1, 0.5, 1, 2],
    'cache_size': [100, 200, 500, 1000]
}

LR_params = {
    'class_weight': ['balanced'],
    'solver': ['newton-cg', 'lbfgs', 'saga'],
    'penalty': ['l2'],
    'C': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100],
    'dual': [False],
    'max_iter': [100, 200, 500, 1000]
}

DT_params = {
    'max_depth': list(range(5, 20)),
    'class_weight': ['balanced'],
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1, 1.0],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15]
}

RF_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'class_weight': ['balanced'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

GB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'subsample': [0.8, 0.9, 1.0]
}

XGB_params = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 10, 20]
}

classifiers_Params = {
    'SVM': SVM_params,
    'Logistic Regression': LR_params,
    'Decision Tree': DT_params,
    'Random Forest': RF_params,
    'Gradient Boosting': GB_params,
    'XGBoost': XGB_params
}


results = pd.DataFrame(columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])


for i, clf in enumerate(classifiers):
    model = clf
    param_grid = classifiers_Params[classifiers_names[i]]
    clf_gs = RandomizedSearchCV(model, param_distributions=param_grid, cv=7, scoring='accuracy', error_score='raise', n_iter=50, random_state=42)
    
    clf_gs.fit(X_train_com_EN, y_train_com_EN)
    best_estimator = clf_gs.best_estimator_
    
    print(best_estimator.get_params())

    pred = best_estimator.predict(X_test_com_EN)
    acs = accuracy_score(y_test_com_EN, pred)
    rec = recall_score(y_test_com_EN, pred)
    prec = precision_score(y_test_com_EN, pred)
    f1 = f1_score(y_test_com_EN, pred)
    cm = confusion_matrix(y_test_com_EN, pred)
    auc = roc_auc_score(y_test_com_EN, best_estimator.predict_proba(X_test_com_EN)[:, 1] if hasattr(best_estimator, "predict_proba") else pred)
    
    results.loc[i] = [classifiers_names[i], acs, rec, prec, f1, cm, auc]

print(results.head())


ensemble_model = VotingClassifier(estimators=[(name, clf) for name, clf in zip(classifiers_names, classifiers)], voting='soft')
ensemble_model.fit(X_train_com_EN, y_train_com_EN)
pred = ensemble_model.predict(X_test_com_EN)
acs = accuracy_score(y_test_com_EN, pred)
rec = recall_score(y_test_com_EN, pred)
prec = precision_score(y_test_com_EN, pred)
f1 = f1_score(y_test_com_EN, pred)
cm = confusion_matrix(y_test_com_EN, pred)
auc = roc_auc_score(y_test_com_EN, ensemble_model.predict_proba(X_test_com_EN)[:, 1] if hasattr(ensemble_model, "predict_proba") else pred)

ensemble_results = pd.DataFrame([['Ensemble', acs, rec, prec, f1, cm, auc]], columns=['estimators', 'accuracy', 'recall_score', 'precision', 'f1_score', 'confusion_matrix', 'roc_auc'])
results = pd.concat([results, ensemble_results])
 
results.head(7)